# Train Hair Segmentation Model

`U-Net` hair segmentation model.


In [1]:
from pathlib import Path
import sys
import time

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch import nn
from torch.utils.data import DataLoader

torch.backends.cudnn.benchmark = True

from systems.static_auto_tryon.auto_app.ml.datasets import HairSegmentationDataset, read_jsonl_manifest
from systems.static_auto_tryon.auto_app.ml.hair_segmentation_model import build_segmentation_model
from systems.static_auto_tryon.auto_app.ml.metrics import dice_coefficient_from_logits, iou_from_logits, pixel_accuracy_from_logits
from systems.static_auto_tryon.auto_app.ml.transforms import ResizeImageAndMask

PROJECT_ROOT


WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
TRAIN_MANIFEST = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'hair_segmentation' / 'train.jsonl'
VAL_MANIFEST = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'hair_segmentation' / 'val.jsonl'

LIGHTWEIGHT_MODE = False
REQUIRE_GPU = True
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'hair_segmentation' / ('v1_unet_lightweight.pt' if LIGHTWEIGHT_MODE else 'v2_unet_product.pt')

MAX_TRAIN_RECORDS = 2000 if LIGHTWEIGHT_MODE else None
MAX_VAL_RECORDS = 400 if LIGHTWEIGHT_MODE else None

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 64 if LIGHTWEIGHT_MODE else 12
EPOCHS = 3 if LIGHTWEIGHT_MODE else 12
LEARNING_RATE = 1e-3 if LIGHTWEIGHT_MODE else 5e-4
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)


In [3]:
train_records = read_jsonl_manifest(TRAIN_MANIFEST)
val_records = read_jsonl_manifest(VAL_MANIFEST)

if MAX_TRAIN_RECORDS is not None:
    train_records = train_records[:MAX_TRAIN_RECORDS]
if MAX_VAL_RECORDS is not None:
    val_records = val_records[:MAX_VAL_RECORDS]

pd.Series(
    {
        'lightweight_mode': LIGHTWEIGHT_MODE,
        'train_records': len(train_records),
        'val_records': len(val_records),
        'device': DEVICE,
        'torch_version': torch.__version__,
        'cuda_version': torch.version.cuda,
    }
)


lightweight_mode           False
train_records              24905
val_records                 4395
device                      cuda
torch_version       2.11.0+cu128
cuda_version                12.8
dtype: object

In [4]:
transform = ResizeImageAndMask(IMAGE_SIZE)
train_dataset = HairSegmentationDataset(train_records, transform=transform)
val_dataset = HairSegmentationDataset(val_records, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,        
    shuffle=True,
    num_workers=4,        
    pin_memory=True       
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

model = build_segmentation_model(model_name='unet', base_channels=32).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()


In [5]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for batch in loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    logits_batches = []
    mask_batches = []
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, masks)
        running_loss += loss.item()
        logits_batches.append(logits)
        mask_batches.append(masks)

    if logits_batches:
        all_logits = torch.cat(logits_batches, dim=0)
        all_masks = torch.cat(mask_batches, dim=0)
    else:
        all_logits = None
        all_masks = None

    metrics = {
        'val_loss': running_loss / max(len(loader), 1),
        'dice': dice_coefficient_from_logits(all_logits, all_masks) if all_logits is not None else 0.0,
        'iou': iou_from_logits(all_logits, all_masks) if all_logits is not None else 0.0,
        'pixel_accuracy': pixel_accuracy_from_logits(all_logits, all_masks) if all_logits is not None else 0.0,
    }
    return metrics


In [6]:
history = []
best_dice = float('-inf')
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    mode_label = 'lightweight' if LIGHTWEIGHT_MODE else 'full'
    print(f'Running epoch {epoch}/{EPOCHS} [{mode_label}]...')

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    metrics = evaluate(model, val_loader, criterion, DEVICE)

    epoch_seconds = time.perf_counter() - epoch_start
    if metrics['dice'] > best_dice:
        best_dice = metrics['dice']
        best_epoch = epoch

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        **metrics,
        'epoch_time': format_seconds(epoch_seconds),
        'best_epoch_so_far': best_epoch,
        'best_dice_so_far': round(best_dice, 4),
    }
    history.append(row)

    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history,
            'last_epoch': epoch,
            'best_epoch': best_epoch,
            'best_dice': best_dice,
        },
        CHECKPOINT_PATH,
    )

    print(f'Completed epoch {epoch}/{EPOCHS}')
    print(row)

print(f'Saved checkpoint to {CHECKPOINT_PATH}')
pd.DataFrame(history)


Running epoch 1/12 [full]...
Completed epoch 1/12
{'epoch': 1, 'train_loss': 0.18210788399899855, 'val_loss': 0.13418768098633685, 'dice': 0.8991715908050537, 'iou': 0.8328282833099365, 'pixel_accuracy': 0.9466041326522827, 'epoch_time': '06:50', 'best_epoch_so_far': 1, 'best_dice_so_far': 0.8992}
Running epoch 2/12 [full]...
Completed epoch 2/12
{'epoch': 2, 'train_loss': 0.11291354785894038, 'val_loss': 0.1095249170604941, 'dice': 0.9211131930351257, 'iou': 0.8645890951156616, 'pixel_accuracy': 0.9576776623725891, 'epoch_time': '24:35', 'best_epoch_so_far': 2, 'best_dice_so_far': 0.9211}
Running epoch 3/12 [full]...
Completed epoch 3/12
{'epoch': 3, 'train_loss': 0.09832981122058308, 'val_loss': 0.0960529534709551, 'dice': 0.9280154705047607, 'iou': 0.8756559491157532, 'pixel_accuracy': 0.9620961546897888, 'epoch_time': '24:33', 'best_epoch_so_far': 3, 'best_dice_so_far': 0.928}
Running epoch 4/12 [full]...
Completed epoch 4/12
{'epoch': 4, 'train_loss': 0.09016115253931181, 'val_los

,epoch,train_loss,val_loss,dice,iou,pixel_accuracy,epoch_time,best_epoch_so_far,best_dice_so_far
0,1,0.182108,0.134188,0.899172,0.832828,0.946604,06:50,1,0.8992
1,2,0.112914,0.109525,0.921113,0.864589,0.957678,24:35,2,0.9211
2,3,0.098330,0.096053,0.928015,0.875656,0.962096,24:33,3,0.9280
3,4,0.090161,0.089842,0.932074,0.882228,0.964936,24:36,4,0.9321
4,5,0.085010,0.088811,0.933935,0.884546,0.964944,24:39,5,0.9339
5,6,0.081053,0.084627,0.936861,0.888911,0.966439,24:38,6,0.9369
6,7,0.077697,0.080822,0.939519,0.893281,0.968056,34:08,7,0.9395
7,8,0.073629,0.082775,0.936950,0.889243,0.966914,48:18,7,0.9395
8,9,0.071521,0.080003,0.939264,0.892754,0.968233,24:37,7,0.9395
9,10,0.068385,0.080129,0.940538,0.894586,0.968102,24:45,10,0.9405
